# Entraînement complet sur Google Colab (GPU)

**Modèles :** CD-CNN et CNN-LSTM-32, avec le même protocole, le même découpage et la même graine.

**Protocole par défaut :**
- Adam, lr fixe 0,001, batch 16, 50 époques max ;
- early stopping sur la validation (10 % des sujets d'entraînement) ;
- GroupKFold 5 plis par sujet ;
- adaptation au sujet : 4 plis 75/25, 5 époques, lr 5e-4, avec les deux modes de z-score (`complet` et `train`).

**Avant de commencer :** `Exécution > Modifier le type d'exécution > GPU T4`.

**Sauvegarde et reprise :**
- Après chaque pli, `train.py` écrit sur Drive (`OUT_DIR`) le json du pli, les poids `.pt` et les résultats d'adaptation.
- Après une déconnexion, il suffit de **relancer toutes les cellules dans l'ordre**. Les plis déjà terminés sont relus, pas recalculés. Un pli interrompu en cours d'entraînement est recommencé depuis le début.
- Le cache des données (`CACHE_DIR`) est sur le disque local de Colab : il est reconstruit en quelques minutes après une déconnexion.

**Non lancés ici (options, seulement sur demande) :** `--hidden 64`, `--pool moyenne`, `--scheduler`, `--clip`, `--norm-essai`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paramètres (à adapter si besoin)

In [ ]:
ZIP_DRIVE = '/content/drive/MyDrive/eegmmidb-1.0.0.zip'   # zip PhysioNet déposé sur Drive
DATA_DIR  = '/content/eegmmidb-1.0.0/files'                 # après décompression
OUT_DIR   = '/content/drive/MyDrive/eeg_resultats'          # résultats + poids, sur Drive
CACHE_DIR = '/content/cache'                                 # cache .npz local (rapide)
REPO_URL  = 'https://github.com/ChaYLbd123/EEG_Classification.git'
REPO_DIR  = '/content/EEG_Classification'
MODELES   = 'cd_cnn cnn_lstm'                                # cnn_lstm = CNN-LSTM-32 (dernier état caché)

## Décompression des données (une fois par session)

In [ ]:
import os
if not os.path.isdir(DATA_DIR):
    # Le zip contient un dossier 'files/' à la racine -> /content/eegmmidb-1.0.0/files
    !unzip -q -n "$ZIP_DRIVE" -d /content/eegmmidb-1.0.0
print(DATA_DIR, ':', len([d for d in os.listdir(DATA_DIR) if d.startswith('S')]), 'dossiers de sujets')

## Code du projet et dépendances

In [ ]:
if os.path.isdir(REPO_DIR):
    !git -C "$REPO_DIR" pull
else:
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
!git log --oneline -1
!pip install -q mne

In [ ]:
import torch
print('PyTorch', torch.__version__)
print('CUDA disponible :', torch.cuda.is_available())
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU"
print('GPU :', torch.cuda.get_device_name(0))

## Vérification des données (attendu : 105 sujets, 0 fichier manquant)

In [ ]:
exclus = {88, 92, 100, 104}
runs = [1, 4, 6, 8, 10, 12, 14]
sujets = [s for s in range(1, 110) if s not in exclus]
manquants = [f"S{s:03d}R{r:02d}" for s in sujets for r in runs
             if not os.path.exists(f"{DATA_DIR}/S{s:03d}/S{s:03d}R{r:02d}.edf")]
print(len(sujets), 'sujets attendus,', len(manquants), 'fichiers manquants', manquants[:5])

## Outil : temps mesuré sur le GPU (à exécuter une fois)

In [ ]:
import json, glob
import numpy as np

def afficher_temps(n_classes, modeles=('cd_cnn', 'cnn_lstm32'), suffixe=''):
    """Temps par époque et par pli, lus dans les json écrits pendant l'entraînement."""
    for m in modeles:
        dossier = f'{OUT_DIR}/{m}_{n_classes}cl{suffixe}'
        plis = sorted(glob.glob(f'{dossier}/pli*_global.json'))
        if not plis:
            print(f'{m} {n_classes} cl : aucun pli terminé')
            continue
        for f in plis:
            g = json.load(open(f, encoding='utf-8'))
            t = g['historique']['temps_s']
            print(f"{m:11s} {n_classes} cl pli {g['pli']} [{g['device']}] : "
                  f"{np.mean(t):.2f} s/époque (méd. {np.median(t):.2f}), {g['epoques_faites']} époques, "
                  f"meilleure {g['meilleure_epoque']}, entraînement {g['temps_entrainement_s']/60:.1f} min")
        for mode in ('complet', 'train'):
            A = sorted(glob.glob(f'{dossier}/pli*_adapt_{mode}.json'))
            if A:
                tt = [json.load(open(a, encoding='utf-8'))['temps_s'] for a in A]
                print(f'{"":11s} adaptation [{mode}] : {np.mean(tt)/60:.1f} min/pli ({len(A)} plis)')

## (Optionnel) Essai rapide sur GPU : 10 sujets, 5 époques

Il vérifie l'installation et mesure le temps sur le GPU en quelques minutes. Ses résultats vont dans des dossiers `*_essai`, séparés des résultats complets.

In [ ]:
!python -W ignore train.py --model $MODELES --classes 2 4 --quick \
    --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"
afficher_temps(2, suffixe='_essai'); afficher_temps(4, suffixe='_essai')

## 1) Tâche 2 classes (gauche / droite) : entraînement complet

105 sujets, 5 plis, CD-CNN puis CNN-LSTM-32. Relancer cette cellule après une déconnexion reprend au premier pli non terminé.

In [ ]:
!python -W ignore train.py --model $MODELES --classes 2 \
    --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"

In [ ]:
afficher_temps(2)

## 2) Tâche 4 classes (repos / gauche / droite / pieds) : entraînement complet

In [ ]:
!python -W ignore train.py --model $MODELES --classes 4 \
    --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"

In [ ]:
afficher_temps(4)

## Résumé (moyenne ± écart-type sur les 5 plis)

In [ ]:
for n in (2, 4):
    for m in ('cd_cnn', 'cnn_lstm32'):
        f = f'{OUT_DIR}/{m}_{n}cl/resume.json'
        if not os.path.exists(f):
            print(f'{m} {n} cl : pas encore de résultat complet')
            continue
        r = json.load(open(f, encoding='utf-8'))
        print(f"{m:11s} {n} cl | global {r['global_moy']:.2%} ± {r['global_std']:.2%} | "
              f"adapt. complet {r['adapt_complet_avant_moy']:.2%} -> {r['adapt_complet_apres_moy']:.2%} | "
              f"adapt. train {r['adapt_train_avant_moy']:.2%} -> {r['adapt_train_apres_moy']:.2%}")